# 📽️ Task 2: Netflix Content Type Prediction Model
### Binary Classification: Predicting 'Movie' vs. 'TV Show' from Metadata

---

## 1. Executive Summary & Problem Formulation
Streaming catalogs feature distinct media formats. Netflix primarily categorizes entries into **Movies** and **TV Shows**.
This task develops a supervised binary classification pipeline that predicts the format of any catalog item based on its metadata:
- **Target Variable**: $y \in \{\text{Movie}, \text{TV Show}\}$
- **Input Features**: `listed_in` (genres), `rating` (audience certificate), `primary_genre`, and `release_year`.


In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sys.path.append(str(Path.cwd().parent))
from src.data_loader import get_preprocessed_data
from src.classifier import ContentTypeClassifier

df = get_preprocessed_data('../data/Dataset.csv')
print("Class Distribution:")
print(df['type'].value_counts())
print("\nClass Proportions:")
print(df['type'].value_counts(normalize=True).round(3))


Class Distribution:
type
Movie      6126
TV Show    2664
Name: count, dtype: int64

Class Proportions:
type
Movie      0.697
TV Show    0.303
Name: proportion, dtype: float64


## 2. Model Training & Comparison
We train and benchmark two distinct algorithmic approaches:
1. **Regularized Logistic Regression**: Linear baseline with $L_2$ regularization.
2. **Random Forest Classifier**: Ensemble of 120 decision trees with depth regularization ($d=14$).


In [2]:
# Train Random Forest Pipeline
rf_model = ContentTypeClassifier(model_type='rf')
rf_results = rf_model.train(df)

# Train Logistic Regression Pipeline
lr_model = ContentTypeClassifier(model_type='lr')
lr_results = lr_model.train(df)

summary_df = pd.DataFrame([
    {"Model": "Random Forest", "Accuracy": rf_results['accuracy'], "Precision": rf_results['precision'], "Recall": rf_results['recall'], "F1-Score": rf_results['f1_score'], "ROC-AUC": rf_results['roc_auc']},
    {"Model": "Logistic Regression", "Accuracy": lr_results['accuracy'], "Precision": lr_results['precision'], "Recall": lr_results['recall'], "F1-Score": lr_results['f1_score'], "ROC-AUC": lr_results['roc_auc']}
])
summary_df


Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC
0        Random Forest    1.0000     1.0000  1.0000    1.0000   1.0
1  Logistic Regression    1.0000     1.0000  1.0000    1.0000   1.0


## 3. Confusion Matrix Analysis


In [3]:
classes = rf_results['classes']
cm = np.array(rf_results['confusion_matrix'])
cm_df = pd.DataFrame(cm, index=[f"Actual {c}" for c in classes], columns=[f"Pred {c}" for c in classes])
print("Random Forest Confusion Matrix:")
print(cm_df)


Random Forest Confusion Matrix:
                Pred Movie  Pred TV Show
Actual Movie          1225             0
Actual TV Show           0           533


## 4. Real-Time Inference Demo
Testing real-world inference with metadata inputs:


In [4]:
test_cases = [
    {"listed_in": "Documentaries, International Movies", "rating": "PG-13", "release_year": 2020},
    {"listed_in": "Crime TV Shows, Docuseries", "rating": "TV-MA", "release_year": 2021},
    {"listed_in": "Children & Family Movies, Comedies", "rating": "TV-Y", "release_year": 2019}
]

for item in test_cases:
    res = rf_model.predict(item)
    print(f"Input: {item['listed_in']} ({item['rating']})")
    print(f" -> Predicted: {res['prediction']} (Confidence: {res['confidence']*100:.1f}%)\n")


Input: Documentaries, International Movies (PG-13)
 -> Predicted: Movie (Confidence: 99.7%)

Input: Crime TV Shows, Docuseries (TV-MA)
 -> Predicted: TV Show (Confidence: 99.8%)

Input: Children & Family Movies, Comedies (TV-Y)
 -> Predicted: Movie (Confidence: 99.4%)


## 5. Architectural Conclusions
1. The presence of format-specific genre phrases (e.g. `Movies` vs `TV Shows` in `listed_in`) provides clear signal, allowing both linear and tree models to achieve near-perfect classification.
2. The model acts as a reliable automated tagger for streaming ingestion pipelines where content type flags may be missing or corrupt.
